In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:f}'.format)


In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem

In [2]:
df = pd.read_csv('antibiotics_enamine.csv')

In [3]:
filtered_df = df[df['druglikeness'] > 0.6]

In [4]:
# A_baumanii
# threshold=0.00, precision=0.01, recall=1.00
# threshold=0.09, precision=0.10, recall=0.61
# threshold=0.28, precision=0.20, recall=0.45
# threshold=0.44, precision=0.30, recall=0.36
# threshold=0.50, precision=0.40, recall=0.34
# threshold=0.57, precision=0.50, recall=0.31
# threshold=0.63, precision=0.60, recall=0.25
# threshold=0.99, precision=0.70, recall=0.07
# threshold=1.00, precision=0.88, recall=0.03
# threshold=1.00, precision=1.00, recall=0.03
# threshold=1.00, precision=1.00, recall=0.03


# P_aeruginosa
# threshold=0.00, precision=0.00, recall=1.00
# threshold=0.02, precision=0.10, recall=0.74
# threshold=0.03, precision=0.20, recall=0.63
# threshold=0.04, precision=0.30, recall=0.54
# threshold=0.05, precision=0.40, recall=0.48
# threshold=0.06, precision=0.50, recall=0.45
# threshold=0.10, precision=0.60, recall=0.43
# threshold=0.18, precision=0.70, recall=0.39
# threshold=0.67, precision=0.80, recall=0.27
# threshold=1.00, precision=1.00, recall=0.02
# threshold=1.00, precision=1.00, recall=0.01
# Neisseria gonorrhoeae
# threshold=0.00, precision=0.03, recall=1.00
# threshold=0.04, precision=0.10, recall=0.79
# threshold=0.08, precision=0.20, recall=0.73
# threshold=0.13, precision=0.30, recall=0.65
# threshold=0.21, precision=0.40, recall=0.55
# threshold=0.33, precision=0.50, recall=0.44
# threshold=0.46, precision=0.60, recall=0.37
# threshold=0.65, precision=0.70, recall=0.27
# threshold=0.84, precision=0.80, recall=0.15
# threshold=0.94, precision=0.90, recall=0.08
# threshold=1.00, precision=1.00, recall=0.00

In [5]:
col_names = ['A_baumanii_avg', 'E_coli_avg', 'K_pneumoniae_avg', 'P_aeruginosa_avg', 'N_gonorrhoeae_avg']
thresholds = [0.18, 0.021, 0.012, 0.014, 0.669]

In [6]:
# ===== Leader (sphere-exclusion) clustering for CV groups =====
# - ECFP bit fingerprints via RDKit's modern generator
# - Invalid SMILES get group = -1 (and are excluded from CV splits)
# - Deterministic by default; set randomize_order=True for alternative valid groupings
from tqdm import tqdm
from typing import List, Optional, Tuple
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from sklearn.model_selection import GroupKFold

# --------- Featurization ---------
def smiles_to_bit_fps(
    smiles: List[str],
    radius: int = 2,
    n_bits: int = 2048,
    use_chirality: bool = True,
) -> Tuple[List[Optional[Chem.Mol]], List[Optional[DataStructs.ExplicitBitVect]], np.ndarray]:
    """
    SMILES -> RDKit Mol -> ECFP (bit) ExplicitBitVect.
    Returns (mols, fps, valid_mask). Invalid SMILES produce mol=None, fp=None.
    """
    fpgen = GetMorganGenerator(radius=radius, fpSize=n_bits, includeChirality=use_chirality)
    mols, fps = [], []
    for s in tqdm(smiles, desc="Featurizing"):
        m = Chem.MolFromSmiles(s)
        if m is None:
            mols.append(None); fps.append(None); continue
        try:
            fp = fpgen.GetFingerprint(m)
        except Exception:
            m, fp = None, None
        mols.append(m); fps.append(fp)
    valid_mask = np.array([f is not None for f in fps], dtype=bool)
    return mols, fps, valid_mask

# ================= Example usage =================
# Assume you have a pandas DataFrame `data_df` with a column 'smi'


In [7]:
folder = '../data/molecules'
for i, col_name in enumerate(col_names):
    subset_df = filtered_df[filtered_df[col_name] > thresholds[i]]
    
    subset_df = subset_df[subset_df['smiles'].apply(lambda x: Chem.MolFromSmiles(x).GetNumAtoms()) > 20]
       
    fps = smiles_to_bit_fps(subset_df['smiles'].to_list())[1]
    from rdkit.SimDivFilters import rdSimDivPickers
    lp = rdSimDivPickers.LeaderPicker()
    thresh = 0.65 # <- minimum distance between cluster centroids
    picks = lp.LazyBitVectorPick(fps, len(fps), thresh)
        
    subset_df = subset_df.iloc[picks]
    subset_df.to_csv(f'{folder}/filtered_{col_name}.csv', index=False)
    print(f'{col_name} has {len(subset_df)} rows')

Featurizing: 100%|██████████| 4780/4780 [00:01<00:00, 4635.09it/s]


A_baumanii_avg has 1189 rows


Featurizing: 100%|██████████| 6905/6905 [00:01<00:00, 4669.36it/s]


E_coli_avg has 1584 rows


Featurizing: 100%|██████████| 5992/5992 [00:01<00:00, 4692.18it/s]


K_pneumoniae_avg has 1381 rows


Featurizing: 100%|██████████| 3955/3955 [00:00<00:00, 4260.31it/s]


P_aeruginosa_avg has 1028 rows


Featurizing: 100%|██████████| 4358/4358 [00:00<00:00, 4442.77it/s]


N_gonorrhoeae_avg has 797 rows


In [13]:
import pandas as pd
pd.set_option('display.float_format', '{:f}'.format)

species_list = ['A_baumanii_avg', 'E_coli_avg', 'K_pneumoniae_avg', 'P_aeruginosa_avg', 'N_gonorrhoeae_avg']
for species in species_list:
    print(species)
    print(df[f'{species}'].describe(percentiles=[0.95, 0.99, 0.999, 0.9998]))
    print()

A_baumanii_avg
count    4210926.000000
mean           0.017967
std            0.024967
min            0.004300
50%            0.009900
95%            0.058400
99%            0.132500
99.9%          0.254000
99.98%         0.324582
max            0.956500
Name: A_baumanii_avg, dtype: float64

E_coli_avg
count    4210926.000000
mean           0.001491
std            0.007312
min            0.000500
50%            0.000800
95%            0.003000
99%            0.011700
99.9%          0.074300
99.98%         0.185182
max            0.987100
Name: E_coli_avg, dtype: float64

K_pneumoniae_avg
count    4210926.000000
mean           0.001110
std            0.005590
min            0.000400
50%            0.000700
95%            0.002400
99%            0.007100
99.9%          0.035000
99.98%         0.100700
max            0.983500
Name: K_pneumoniae_avg, dtype: float64

P_aeruginosa_avg
count    4210926.000000
mean           0.002978
std            0.005311
min            0.001100
50%         